In [ ]:
#Parsing Data

In [ ]:
# Process Data -> 지표표함 data

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

def parse_volume(x):
    """'1.61K', '71.47M', '2.3B' 등 단위를 숫자로 변환"""
    s = str(x).strip()
    if s.endswith('M'):
        return float(s[:-1]) * 1_000_000
    if s.endswith('K'):
        return float(s[:-1]) * 1_000
    if s.endswith('B'):
        return float(s[:-1]) * 1_000_000_000
    try:
        return float(s.replace(',', ''))
    except:
        return np.nan

# 1) 원본 Back Test 루트 폴더 & 결과를 저장할 Processed Data 폴더
# C:\Users\LabPC\OneDrive\
    
BACK_TEST_ROOT   = r"C:\Users\LabPC\OneDrive\주식\Back Test"
PROCESSED_FOLDER = r"C:\Users\LabPC\OneDrive\주식\Processed Data"

os.makedirs(PROCESSED_FOLDER, exist_ok=True)

# 2) 각 종목(하위 폴더)마다 CSV 파일 찾아서 처리
for company in os.listdir(BACK_TEST_ROOT):
    company_path = os.path.join(BACK_TEST_ROOT, company)
    if not os.path.isdir(company_path):
        continue

    csv_files = glob.glob(os.path.join(company_path, "*.csv"))
    for file_path in csv_files:
        # 3) CSV 불러와서 컬럼명 한글로 변경
        df = (
            pd.read_csv(file_path)
              .rename(columns={
                  'Date':     '날짜',
                  'Price':    '종가',
                  'Open':     '시가',
                  'High':     '고가',
                  'Low':      '저가',
                  'Vol.':     '거래량',
                  'Change %': '변동 %'
              })
        )

        # 4) 날짜 파싱 & 정렬
        df['날짜'] = pd.to_datetime(df['날짜'], format='%m/%d/%Y')
        df = df.sort_values('날짜').reset_index(drop=True)

        # 4.1) 가격 컬럼(종가, 시가, 고가, 저가) 콤마 제거 후 float 변환
        for col in ['종가', '시가', '고가', '저가']:
            df[col] = (
                df[col]
                  .astype(str)
                  .str.replace(',', '', regex=False)
                  .astype(float)
            )

        # ───────────────────────────────────────────────
        #  추가: 데이터 시작·종료일 추출
        # ───────────────────────────────────────────────
        start_date = df['날짜'].min().strftime('%Y-%m-%d')
        end_date   = df['날짜'].max().strftime('%Y-%m-%d')
        df['시작일'] = start_date
        df['종료일'] = end_date

        # 5) 거래량·변동 % 숫자 처리
        df['거래량'] = df['거래량'].apply(parse_volume)
        df['변동 %'] = (
            df['변동 %']
              .astype(str)
              .str.replace(',', '', regex=False)
              .str.rstrip('%')
              .astype(float)
        )

        # === 지표 계산 ===

        # ▶ RSI (14일)
        delta     = df['종가'].diff()
        gain      = delta.clip(lower=0)
        loss      = -delta.clip(upper=0)
        avg_gain  = gain.rolling(window=14).mean()
        avg_loss  = loss.rolling(window=14).mean()
        df['RSI (14일)'] = 100 - (100 / (1 + avg_gain/avg_loss))

        # ▶ Bollinger Bands (20일)
        m = df['종가'].rolling(window=20).mean()
        s = df['종가'].rolling(window=20).std()
        df['볼린저밴드 상단'] = m + 2 * s
        df['볼린저밴드 하단'] = m - 2 * s

        # ▶ MACD & Signal
        ema12 = df['종가'].ewm(span=12, adjust=False).mean()
        ema26 = df['종가'].ewm(span=26, adjust=False).mean()
        df['MACD']        = ema12 - ema26
        df['MACD 시그널'] = df['MACD'].ewm(span=9, adjust=False).mean()

        # ▶ SMA (5,10,20,60,120,200일)
        for p in [5, 10, 20, 60, 120, 200]:
            df[f"SMA {p}일"] = df['종가'].rolling(window=p).mean()

        # ▶ 가격·거래량 % 변화 (2주,3개월,6개월,1년)
        periods = {'2주': 10, '3개월': 63, '6개월': 126, '1년': 252}
        for label, span in periods.items():
            df[f'가격 상승률 ({label})']   = df['종가'].pct_change(span)  * 100
            df[f'거래량 상승률 ({label})'] = df['거래량'].pct_change(span) * 100

        # 6) 저장 (회사명_원본이름_지표포함.csv)
        base      = os.path.splitext(os.path.basename(file_path))[0]
        save_name = f"{company}_{base}_지표포함.csv"
        save_path = os.path.join(PROCESSED_FOLDER, save_name)

        # ───────────────────────────────────────────────
        # ★ 중복 제거: 같은 회사명_원본이름_지표포함.csv 패턴의 이전 파일 삭제
        # ───────────────────────────────────────────────
        pattern = os.path.join(PROCESSED_FOLDER, f"{company}_*_지표포함.csv")
        for old_file in glob.glob(pattern):
            try:
                os.remove(old_file)
            except OSError:
                pass  # 삭제 실패해도 무시

        # 새로운 파일 저장
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        print(f"✅ Processed: {company} → {save_name} (기간: {start_date} ~ {end_date})")


In [ ]:
#OPtimization or Simulation

In [ ]:
# import os, glob, argparse
# from datetime import datetime
# import pandas as pd, numpy as np, optuna
# from optuna.importance import get_param_importances

# # ─── Shared: 데이터 로드 & 신호 & 백테스트 ─────────────────
# def load_data(path, macro_dir, start, end):
#     df = pd.read_csv(path, encoding='utf-8-sig', parse_dates=['날짜'])
#     df = df[(df.날짜>=start)&(df.날짜<end)].reset_index(drop=True)
#     vix_files = glob.glob(os.path.join(macro_dir, "*VIX*.csv"))
#     if vix_files:
#         vix = pd.read_csv(vix_files[0], parse_dates=['날짜'], encoding='utf-8-sig')
#         df = df.merge(vix[['날짜','VIX']], on='날짜', how='left')
#     else:
#         df['VIX'] = np.nan
#     return df

# RULES = {
#     'macro_buy':  lambda r,p: not np.isnan(r.VIX) and r.VIX>=p['vix_buy_th'],
#     'macro_sell': lambda r,p: not np.isnan(r.VIX) and r.VIX<=p['vix_sell_th'],
#     'rule_buy':   lambda r,p: (
#         r['RSI (14일)']<p['rsi_buy_th'] and
#         r['종가']<r['볼린저밴드 하단']*(1+p['boll_buffer']) and
#         r['MACD']>r['MACD 시그널'] and
#         r['SMA 5일']>r['SMA 10일']<r['SMA 60일'] and
#         r['가격 상승률 (2주)']<p['tw_price_th'] and r['가격 상승률 (3개월)']<p['tm_price_th'] and
#         r['거래량 상승률 (2주)']<p['tw_vol_th'] and r['거래량 상승률 (3개월)']<p['tm_vol_th']
#     ),
#     'rule_sell':  lambda r,p: r['RSI (14일)']>p['rsi_sell_th']
# }

# PARAM_BOUNDS = {
#     'vix_buy_th':(0,100),'vix_sell_th':(0,100),'rsi_buy_th':(0,100),
#     'boll_buffer':(0,0.1),'tw_price_th':(0,20),'tm_price_th':(0,50),
#     'tw_vol_th':(0,100),'tm_vol_th':(0,300),'rsi_sell_th':(0,100)
# }

# def backtest(df, p, extra=False, cooldown=0):
#     cash, shares, total, last = 10000, 0, 10000, None
#     for r in df.itertuples():
#         price, date = r.종가, r.날짜
#         ok = last is None or (date-last).days>=cooldown
#         if shares==0 and RULES['macro_buy'](r,p) and ok:
#             if extra: cash+=10000; total+=10000
#             shares, cash, last = cash/price, 0, date
#         elif shares>0 and RULES['macro_sell'](r,p):
#             cash, shares, last = shares*price, 0, None
#         elif shares==0 and RULES['rule_buy'](r,p) and ok:
#             if extra: cash+=10000; total+=10000
#             shares, cash, last = cash/price, 0, date
#         elif shares>0 and RULES['rule_sell'](r,p):
#             cash, shares, last = shares*price, 0, None
#     final = cash + shares*df.iloc[-1].종가
#     return (final-total)/total*100

# # ─── 최적화 함수 ─────────────────────────────────────
# def optimize(data_dir, macro_dir, start, end):
#     companies = sorted({os.path.basename(f).split('_')[0] for f in glob.glob(f"{data_dir}/*_지표포함.csv")})
#     # 종목 선택
#     print("🔔 최적화 가능 종목:")
#     for i, c in enumerate(companies,1): print(f"  {i}. {c}")
#     sel = input("선택(번호 or all): ").strip()
#     idxs = range(len(companies)) if sel=='all' else [int(x)-1 for x in sel.split(',')]
#     recs = []
#     for i in idxs:
#         comp = companies[i]
#         df = load_data(glob.glob(f"{data_dir}/{comp}_*_지표포함.csv")[0], macro_dir, start, end)
#         # 1) TPE 최적화
#         def obj_tpe(trial):
#             p={k:trial.suggest_float(k,*b) for k,b in PARAM_BOUNDS.items()}
#             if p['vix_sell_th']>p['vix_buy_th']: p['vix_sell_th']=p['vix_buy_th']
#             return backtest(df,p)
#         tpe = optuna.create_study(direction='maximize'); tpe.optimize(obj_tpe, n_trials=500)
#         best = tpe.best_params
#         # 2) CMA-ES 재탐색
#         imp = [k for k,v in get_param_importances(tpe).items() if v>0.05]
#         def obj_cma(trial):
#             p=best.copy()
#             for k in imp:
#                 lo,hi=PARAM_BOUNDS[k]; d=0.2*(hi-lo)
#                 p[k]=trial.suggest_float(k,max(lo,best[k]-d),min(hi,best[k]+d))
#             if p['vix_sell_th']>p['vix_buy_th']: p['vix_sell_th']=p['vix_buy_th']
#             return backtest(df,p)
#         cma = optuna.create_study(direction='maximize', sampler=optuna.samplers.CmaEsSampler())
#         cma.optimize(obj_cma, n_trials=200); best.update(cma.best_params)
#         recs.append({'종목':comp,'Start':start,'End':end,'ROI(%)':round(cma.best_value,2),**best})
#     # 파일 저장
#     dfp=pd.DataFrame(recs); dfp.index+=1
#     os.makedirs(os.path.join(args.out,'Parameters'),exist_ok=True)
#     dfp.to_excel(os.path.join(args.out,'Parameters','parameters.xlsx'),index_label='Index')

# # ─── 시뮬레이션 함수 ─────────────────────────────────
# def simulate(params_file, data_dir, macro_dir, out_dir):
#     os.makedirs(out_dir, exist_ok=True)
#     dfp = pd.read_excel(params_file)
#     # Index 선택
#     print("🔔 시뮬레이션 가능 Index:")
#     for _,r in dfp.iterrows():
#         print(f"  {int(r['Index'])}. {r['종목']} ({r['Start']}~{r['End']}, ROI: {r['ROI(%)']}%)")
#     sel = input("시뮬레이션할 Index 번호(콤마 or all): ").strip()
#     if sel.lower()=='all':
#         selected = dfp.copy()
#     else:
#         nums = [int(x) for x in sel.split(',') if x.strip().isdigit()]
#         selected = dfp[dfp['Index'].isin(nums)].copy()
#     # 시뮬레이션 실행
#     for _,r in selected.iterrows():
#         comp, s, e = r['종목'], r['Start'], r['End']
#         params = r.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt'],errors='ignore').to_dict()
#         df = load_data(glob.glob(f"{data_dir}/{comp}_*_지표포함.csv")[0], macro_dir, s, e)
#         for extra in (False, True):
#             roi = backtest(df, params, extra)
#             fname = f"{int(r['Index'])}_{comp}_{'extra' if extra else 'once'}_{s}_{e}_ROI_{roi:.2f}.csv"
#             pd.DataFrame().to_csv  # 기존 로깅 캡처 로직 삽입 가능
#             # 실제 로그 저장 코드로 교체

# # ─── CLI 진입점 ───────────────────────────────────────
# if __name__=='__main__':
#     parser=argparse.ArgumentParser();
#     parser.add_argument('mode',choices=['optimize','simulate']);
#     parser.add_argument('--data',default=r'D:\주식\Processed Data');
#     parser.add_argument('--macro',default=r'D:\주식\Macro Data');
#     parser.add_argument('--out',default=r'D:\주식\Results');
#     args=parser.parse_args()
#     if args.mode=='optimize':
#         start=input("시작일(YYYY-MM-DD, 기본 2022-01-01):") or '2022-01-01'
#         end=input(f"종료일(YYYY-MM-DD, 기본 {datetime.now().strftime('%Y-%m-%d')}):") or datetime.now().strftime('%Y-%m-%d')
#         optimize(args.data, args.macro, start, end)
#     else:
#         simulate(os.path.join(args.out,'Parameters','parameters.xlsx'), args.data, args.macro, args.out)


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# 경로 설정 C:\Users\LabPC\OneDrive\
# ─────────────────────────────────────────────────────────────
DATA_DIR     = r"C:\Users\LabPC\OneDrive\주식\Processed Data"
MACRO_DIR    = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
RESULTS_ROOT = r"C:\Users\LabPC\OneDrive\주식\Results"
PARAM_FILE   = os.path.join(RESULTS_ROOT, "Parameters", "parameters.xlsx")

os.makedirs(RESULTS_ROOT, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 신호 함수
# ─────────────────────────────────────────────────────────────
def is_macro_buy(r, p): return not np.isnan(r['VIX']) and r['VIX'] >= p['vix_buy_th']
def is_macro_sell(r, p): return not np.isnan(r['VIX']) and r['VIX'] <= p['vix_sell_th']
def is_rule_buy(r, p):
    return (
        r['RSI (14일)'] < p['rsi_buy_th'] and
        r['종가'] < r['볼린저밴드 하단']*(1+p['boll_buffer']) and
        r['MACD'] > r['MACD 시그널'] and
        r['SMA 5일'] > r['SMA 10일'] and r['SMA 5일'] < r['SMA 60일'] and
        r['가격 상승률 (2주)'] < p['tw_price_th'] and r['가격 상승률 (3개월)'] < p['tm_price_th'] and
        r['거래량 상승률 (2주)'] < p['tw_vol_th'] and r['거래량 상승률 (3개월)'] < p['tm_vol_th']
    )
def is_rule_sell(r, p): return r['RSI (14일)'] > p['rsi_sell_th']

# ─────────────────────────────────────────────────────────────
# 백테스트 함수
# ─────────────────────────────────────────────────────────────
def run_backtest(df, params, extra_on_buy=False, cooldown_days=0):
    cash, shares, total = 10000.0, 0.0, 10000.0
    logs, last = [], None
    for _, r in df.iterrows():
        date, price, vix = r['날짜'], r['종가'], r.get('VIX', np.nan)
        ok = last is None or (date-last).days>=cooldown_days
        if is_macro_buy(r, params) and ok:
            if extra_on_buy:
                cash += 10000.0; total += 10000.0
            shares, cash, last = cash/price, 0.0, date
            logs.append([date, f"BUY_MACRO_VIX (VIX={vix:.2f})", price, shares, cash, cash+shares*price, total])
        elif shares>0 and is_macro_sell(r, params):
            cash, shares, last = shares*price,0,None
            logs.append([date, f"SELL_MACRO_VIX (VIX={vix:.2f})", price, shares, cash, cash, total])
        elif is_rule_buy(r, params) and ok:
            if extra_on_buy:
                cash += 10000.0; total += 10000.0
            shares, cash, last = cash/price, 0.0, date
            logs.append([date, "BUY_RULE", price, shares, cash, cash+shares*price, total])
        elif shares>0 and is_rule_sell(r, params):
            cash, shares, last = shares*price,0,None
            logs.append([date, "SELL_RULE", price, shares, cash, cash, total])
    if shares>0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        cash += shares*price; shares=0
        logs.append([date, "LIQUIDATE", price, shares, cash, cash, total])
    cols=["날짜","액션","가격","보유주","현금","총자산","투입금액"]
    df_logs=pd.DataFrame(logs,columns=cols)
    df_logs["ROI(%)"]=(df_logs["총자산"]/df_logs["투입금액"]*100).round(2)
    return df_logs

# ─────────────────────────────────────────────────────────────
# 최적화 스텁(필요시 구현)
# ─────────────────────────────────────────────────────────────
def optimize():
    print("Optimize flow not implemented here.")

# ─────────────────────────────────────────────────────────────
# 시뮬레이션 (simulate_all.py 동일 구현)
# ─────────────────────────────────────────────────────────────
def simulate():
    dfp = pd.read_excel(PARAM_FILE)
    print("🔔 시뮬레이션 가능 인덱스 목록:")
    for _, row in dfp.iterrows():
        idx, comp, s, e, roi = int(row['Index']), row['종목'], row['Start'], row['End'], row['ROI(%)']
        print(f"  {idx}. {comp} ({s} ~ {e}, ROI: {roi:.2f}%)")
    sel = input("시뮬레이션할 Index(콤마 or all): ").strip()
    selected = dfp if sel=='all' else dfp[dfp.Index.isin([int(x) for x in sel.split(',')])]
    custom_start, custom_end = [], []
    for _, row in selected.iterrows():
        comp = row['종목']
        print(f"\n📌 종목: {comp}")
        s = input("  시작일 입력 (YYYY-MM-DD): ").strip()
        e = input("  종료일 입력 (YYYY-MM-DD): ").strip()
        custom_start.append(pd.to_datetime(s))
        custom_end.append(pd.to_datetime(e))
    selected['Start'] = custom_start
    selected['End']   = custom_end
    print("\n▶ 선택 및 사용자 지정 날짜:")
    print(selected[['Index','종목','Start','End','ROI(%)']].to_string(index=False))
    for _, row in selected.iterrows():
        idx, comp, start, end = int(row['Index']), row['종목'], row['Start'], row['End']
        # load & filter
        df_raw = pd.read_csv(glob.glob(os.path.join(DATA_DIR, f"{comp}_*_지표포함.csv"))[0], encoding='utf-8-sig')
        df_raw['날짜'] = pd.to_datetime(df_raw['날짜'])
        df = df_raw[(df_raw['날짜'] >= start) & (df_raw['날짜'] <= end)].reset_index(drop=True)
        # VIX merge
        vix_files = glob.glob(os.path.join(MACRO_DIR, "*VIX*.csv"))
        if vix_files:
            vix = pd.read_csv(vix_files[0], parse_dates=[0], encoding='utf-8-sig', header=0)
            vix.columns = ['날짜','VIX']
            df = df.merge(vix, on='날짜', how='left')
        else:
            df['VIX'] = np.nan
        out_dir = os.path.join(RESULTS_ROOT, comp)
        os.makedirs(out_dir, exist_ok=True)
        # ① once
        df_once = run_backtest(df, row.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt']).to_dict(), False, 0)
        roi_once = df_once['ROI(%)'].iloc[-1]
        fname1 = f"{idx}_{comp}_once_{start.date()}_{end.date()}_ROI_{roi_once:.2f}.csv"
        df_once.to_csv(os.path.join(out_dir, fname1), index=False, encoding='utf-8-sig')
        print(f"✅ {fname1}")
        # ② extra
        df_extra = run_backtest(df, row.drop(['Index','종목','Start','End','ROI(%)','OptimizedAt']).to_dict(), True, 0)
        roi_extra = df_extra['ROI(%)'].iloc[-1]
        fname2 = f"{idx}_{comp}_extra_{start.date()}_{end.date()}_ROI_{roi_extra:.2f}.csv"
        df_extra.to_csv(os.path.join(out_dir, fname2), index=False, encoding='utf-8-sig')
        print(f"✅ {fname2}")
        # ③ baseline1
        min_p, max_p = df['종가'].min(), df['종가'].max()
        roi_b1 = (max_p/min_p - 1)*100
        df_b1 = pd.DataFrame([
            [df.loc[df['종가'].idxmin(), '날짜'], f"BUY at {min_p:.2f}", min_p, 10000/min_p, np.nan, 10000, f"{roi_b1:.2f}%"],
            [df.loc[df['종가'].idxmax(), '날짜'], f"SELL at {max_p:.2f}", max_p, 0.0, (10000/min_p)*max_p, (10000/min_p)*max_p, f"{roi_b1:.2f}%"]
        ], columns=["날짜","액션","가격","보유주","현금","총자산","ROI(%)"])
        fname3 = f"{idx}_{comp}_baseline1_{start.date()}_{end.date()}_ROI_{roi_b1:.2f}.csv"
        df_b1.to_csv(os.path.join(out_dir, fname3), index=False, encoding='utf-8-sig')
        print(f"✅ {fname3}")
        # ④ DCA
        days = (end - start).days
        total_inj = days * 10000.0
        daily_amt = total_inj / days
        logs = []
        for i, r in df.iterrows():
            date, price = r['날짜'], r['종가']
            shares = (i+1) * daily_amt / price
            logs.append([date, "DCA_BUY", price, shares, daily_amt, total_inj])
        df_dca = pd.DataFrame(logs, columns=["날짜","액션","가격","보유주","투입금액","총투입"])
        df_dca["총자산"] = df_dca["보유주"] * df_dca["가격"]
        df_dca["ROI(%)"] = ((df_dca["총자산"]/df_dca["총투입"] - 1)*100).round(2)
        fname4 = f"{idx}_{comp}_dca_{start.date()}_{end.date()}_ROI_{df_dca['ROI(%)'].iloc[-1]:.2f}.csv"
        df_dca.to_csv(os.path.join(out_dir, fname4), index=False, encoding='utf-8-sig')
        print(f"✅ {fname4}")

# ─────────────────────────────────────────────────────────────
def main():
    mode = input('mode(optimize/simulate): ') or 'simulate'
    if mode == 'optimize':
        optimize()
    else:
        simulate()

if __name__=='__main__': main()
